# Tutorial 05 — EXIT Charts

An EXIT (Extrinsic Information Transfer) curve plots the *mutual information* of a component's extrinsic LLR output `I_E` against the mutual information of its a-priori input `I_A`. Two components iteratively exchange LLRs converge if and only if their curves do not cross between `(0,0)` and `(1,1)` — the *EXIT tunnel*.

This tutorial computes the modem curve for MS-PRS at one Eb/N0 and overlays the rate-½ K=3 decoder curve.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nsm.modem.msprs import precompute as nsm_pre, exit_curve_nsm, modulate
from nsm.codec.conv import precompute as conv_pre, encode
from nsm.exit import exit_curve as decoder_exit_curve
from nsm.channel.awgn import setup, transmit

In [ ]:
L0, SRC = 3, 4096; CODER = {'K': 3, 'octal_code': (0o5, 0o7)}
code = conv_pre(CODER, SRC); nsm = nsm_pre(L0, code['coding_length'], 'unbalanced')
ch   = setup((3, 3, 1), 0.5, rate=0.5); nv, ns = ch['noise_var'][0], ch['noise_std'][0]
rng  = np.random.default_rng(0)
src   = rng.integers(0, 2, SRC).astype(np.int32)
tail  = np.concatenate([src, np.zeros(code['memory'], dtype=np.int32)])
coded = encode(tail, code['coding_length'], code['polynomials'])
rx    = transmit(modulate(coded, L0, nsm['h0'], nsm['h1']), ns)

## Modem EXIT curve (parallel JIT kernel)

In [ ]:
IA = np.linspace(0.05, 0.95, 12)
ie_modem, _, _, _ = exit_curve_nsm(IA, coded, rx, nv, nsm['modulation_length'],
                                    nsm['branch_labels'], code['coding_length'],
                                    nsm['memory'], nsm['total_states'],
                                    nsm['next_states'], nsm['branch_indices'], 5)

## Decoder EXIT curve

In [ ]:
ie_dec, _, _ = decoder_exit_curve(IA, coded, code['coding_length'], SRC,
                                    code['n_outputs'], code['memory'],
                                    code['total_states'], code['next_states'],
                                    code['outputs'], 5)

## The convergence tunnel
The decoder is plotted with axes swapped so its x-axis is its *output* and its y-axis is its *input*: the two curves are directly stackable along a turbo trajectory.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(IA, ie_modem, 'o-', label='MS-PRS modem (L0=3 unbalanced)')
ax.plot(ie_dec, IA, 's-', label='conv decoder (K=3, rate ½)')
ax.plot([0, 1], [0, 1], 'k:', alpha=0.5)
ax.set_xlabel(r'$I_A$ (modem) / $I_E$ (decoder)')
ax.set_ylabel(r'$I_E$ (modem) / $I_A$ (decoder)')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal'); ax.grid(alpha=0.3)
ax.legend(loc='lower right', fontsize='small')
ax.set_title('EXIT chart at Eb/N0 = 3 dB')